# 00 - Ingest Stock Raw Seed
Notebook này tải dữ liệu từ yfinance và lưu vào bảng Delta:
`stock_demo.cloud_stock_raw_seed`

Bảng này sẽ là input cho Lakeflow / Delta Live Tables pipeline graph.

In [0]:
%pip install yfinance pandas pyarrow

In [0]:
dbutils.library.restartPython()

In [0]:
import yfinance as yf
import pandas as pd
from datetime import datetime

# Có thể sửa watchlist ở đây
tickers = [
    # US Stocks
    "AAPL", "MSFT", "GOOGL", "AMZN", "META",
    "NVDA", "TSLA", "NFLX", "AMD", "INTC",

    # ETFs / Index proxies
    "SPY", "QQQ",

    # Crypto
    "BTC-USD", "ETH-USD",

    # Forex
    "EURUSD=X", "JPY=X",

    # Commodities
    "GC=F",   # Gold Futures
    "CL=F",   # Crude Oil Futures

    # Vietnam-related / VinFast
    "VFS",

    # Banking / Finance
    "JPM" ]

period = "1y"
interval = "1d"

all_data = []

for ticker in tickers:
    print(f"Downloading {ticker}...")
    df = yf.download(
        ticker,
        period=period,
        interval=interval,
        auto_adjust=False,
        progress=False
    )

    if df is None or df.empty:
        print(f"Skip {ticker}: no data")
        continue

    # Trường hợp yfinance trả MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        # Nếu chỉ có 1 ticker trong mỗi lượt download, lấy level đầu tiên: Open, High, Low...
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

    df = df.reset_index()

    # Chuẩn hóa tên cột Date/Datetime
    if "Datetime" in df.columns and "Date" not in df.columns:
        df = df.rename(columns={"Datetime": "Date"})

    df["Ticker"] = ticker
    df["IngestedAt"] = datetime.utcnow()

    keep_cols = ["Date", "Ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume", "IngestedAt"]
    existing_cols = [c for c in keep_cols if c in df.columns]
    df = df[existing_cols]

    all_data.append(df)

if not all_data:
    raise ValueError("No data downloaded from yfinance. Check tickers or internet access.")

final_df = pd.concat(all_data, ignore_index=True)

# Chuẩn hóa tên cột để SQL dễ đọc
final_df = final_df.rename(columns={"Adj Close": "Adj_Close"})

# Ép kiểu an toàn
final_df["Date"] = pd.to_datetime(final_df["Date"]).dt.date
for col in ["Open", "High", "Low", "Close", "Adj_Close"]:
    if col in final_df.columns:
        final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

if "Volume" in final_df.columns:
    final_df["Volume"] = pd.to_numeric(final_df["Volume"], errors="coerce").fillna(0).astype("int64")

display(final_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stock_demo")

spark_df = spark.createDataFrame(final_df)

spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("stock_demo.cloud_stock_raw_seed")

print("Saved table: stock_demo.cloud_stock_raw_seed")
display(spark.table("stock_demo.cloud_stock_raw_seed"))

In [0]:
%sql
SELECT Ticker, COUNT(*) AS rows, MIN(Date) AS min_date, MAX(Date) AS max_date
FROM stock_demo.cloud_stock_raw_seed
GROUP BY Ticker
ORDER BY Ticker;